# 2.3. GIẢI THUẬT 3: BAYES NGÂY THƠ (NAÏVE BAYES)


# 2.3.2. BÀI TOÁN 1 : BAYES NGÂY THƠ (NAÏVE BAYES)

**Môn:** Máy học – Khoa Công nghệ thông tin, Trường Đại học Sài Gòn  
**Bài toán 1:** Phát hiện thư rác (spam) – phân loại tin nhắn thành **ham** (bình thường) hoặc **spam**.  
**Dữ liệu:** [Email Spam Detection (Kaggle)](https://www.kaggle.com/code/zabihullah18/email-spam-detection) – file `spam.csv` (cột `v1` = nhãn, `v2` = nội dung).

> **Cách chạy:** tải `spam.csv` từ Kaggle, đặt cùng thư mục với notebook (hoặc chạy trên Kaggle).

**Ý tưởng:** biến văn bản thành vector đếm từ (Bag-of-Words) bằng `CountVectorizer`, rồi dùng `MultinomialNB` – biến thể Naive Bayes phù hợp dữ liệu đếm số lần xuất hiện của từ.

## 2.3.2. BÀI TOÁN 1 : Nhiệm vụ 1 – Phân loại sử dụng Naïve Bayes
### Bước 1 – Import thư viện và nạp dữ liệu

In [ ]:
#Importing the Necessary libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import os
path = next((p for p in ['spam.csv', '/kaggle/input/email-spam-classification-dataset-csv/spam.csv',
                         '/kaggle/input/sms-spam-collection-dataset/spam.csv'] if os.path.exists(p)), 'spam.csv')
data = pd.read_csv(path, encoding='latin-1')
#display the first 5 rows
data.head()

### Bước 2 – Xử lý dữ liệu trước khi xây dựng mô hình
- Bỏ 3 cột thừa toàn giá trị `NaN` (`Unnamed: 2, 3, 4`). *(Tài liệu viết thêm `axis=1` cùng với `columns=`; pandas phiên bản mới báo lỗi khi truyền cả hai nên đã bỏ `axis=1`.)*
- Đổi tên 2 cột còn lại thành `label` và `text` cho dễ đọc.
- Tách đặc trưng `X` (văn bản) và nhãn `y`, rồi chia 80% train – 20% test.

In [ ]:
# Drop the columns with NaN values
data = data.drop(columns=['Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'])
# Rename columns for clarity:
data.columns = ['label', 'text']
# Separate features (X) and target labels (y)
X = data.drop('label', axis=1)
y = data['label']
# Split the data into training and testing sets (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(data['label'].value_counts())
print('Train:', X_train.shape, '| Test:', X_test.shape)

### Bước 3 – Vector hóa nội dung HAM | SPAM của tập train và tập test
`fit_transform` trên tập train (học từ điển từ vựng và đếm từ); chỉ `transform` trên tập test (dùng lại từ điển của tập train, tránh rò rỉ dữ liệu).

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()
# Fit and transform the training data (X_train)
X_train_vectorized = vectorizer.fit_transform(X_train['text'])
# Transform the test data (X_test)
X_test_vectorized = vectorizer.transform(X_test['text'])
print('Kích thước từ điển:', len(vectorizer.vocabulary_), 'từ')
print('Ma trận train:', X_train_vectorized.shape)

### Bước 4 – Xây dựng mô hình Naïve Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB
classifier = MultinomialNB()
classifier.fit(X_train_vectorized, y_train)

### Bước 5 – Đánh giá hiệu quả mô hình

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
# Make predictions on the test data
y_pred = classifier.predict(X_test_vectorized)
# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
classification_rep = classification_report(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")
print("Confusion Matrix:")
print(conf_matrix)
print("Classification Report:")
print(classification_rep)

---
## Mở rộng
### a) Những từ "đặc trưng spam" nhất
Mô hình lưu log xác suất của từng từ theo mỗi lớp (`feature_log_prob_`). Hiệu `log P(từ|spam) − log P(từ|ham)` càng lớn thì từ càng đặc trưng cho spam.

In [ ]:
words = np.array(vectorizer.get_feature_names_out())
idx_spam = list(classifier.classes_).index('spam')
idx_ham = list(classifier.classes_).index('ham')
diff = classifier.feature_log_prob_[idx_spam] - classifier.feature_log_prob_[idx_ham]
print('Top 15 từ nghiêng về SPAM:', list(words[np.argsort(diff)[::-1][:15]]))
print('Top 15 từ nghiêng về HAM :', list(words[np.argsort(diff)[:15]]))

### b) Dự đoán tin nhắn mới
Muốn dự đoán văn bản mới, phải đưa qua **cùng** `vectorizer` đã huấn luyện.

In [ ]:
new_messages = ['Congratulations! You have won a free prize. Call now to claim your reward',
                'Hi, are we still meeting for lunch tomorrow?']
pred = classifier.predict(vectorizer.transform(new_messages))
proba = classifier.predict_proba(vectorizer.transform(new_messages))
for m, p, pr in zip(new_messages, pred, proba):
    print(f'{p.upper():5s} (P={pr.max():.3f}) <- {m}')